# 02 — Monthly Counts & Period Definitions (Stage 2 & 3)

### What this notebook does:
1. **Queries monthly document counts** from Arctic Shift (without saving any full text).
2. **Estimates token volumes** per month and quarter for your target subreddits or all of Reddit.
3. **Triage check:** Verifies if each subreddit has sufficient data (e.g. >5 million tokens) to train reliable embeddings.
4. **Freezes `period_definitions.csv`:** Creates the exact list of time periods that the training pipeline will run.

---

In [ ]:
# =============================================================================
# Cell 1 — USER SETTINGS & PERIOD PLAN (EDIT THIS CELL)
# =============================================================================

# 1. CHOOSE YOUR SCOPE MODE:
#    - "SUBREDDIT_LIST" : Train embeddings for specific subreddits
#    - "ALL_REDDIT"     : Train embeddings across the entire platform (all Reddit combined)
CORPUS_MODE = "SUBREDDIT_LIST"  # Options: "SUBREDDIT_LIST" or "ALL_REDDIT"

# 2. SUBREDDIT SELECTION (Only used if CORPUS_MODE == "SUBREDDIT_LIST"):
#    - Set to None to automatically read from 'config/subreddit_list.csv'
#    - OR provide a list of subreddit names directly: e.g. ["AskAcademia", "PhD", "academia"]
CUSTOM_SUBREDDITS = None

# 3. DATE RANGE:
#    - True  : Process full historical range (2013-01 to 2025-12)
#    - False : Process 3 sample probe months only (fast check)
FULL_RANGE = False
PROBE_MONTHS = ["2015-06", "2019-01", "2023-07"]

# 4. FREEZE PERIODS GATE:
#    - False : Generates counts and triage table for your review.
#    - True  : Writes and locks 'config/period_definitions.csv' for the training pipeline.
FREEZE_PERIODS = False

print(f"Settings: Mode={CORPUS_MODE} | Range={'FULL (2013-2025)' if FULL_RANGE else PROBE_MONTHS} | Freeze={FREEZE_PERIODS}")

In [ ]:
# Bootstrap: locate the repo root and add it to sys.path so the shared 'src'
# package is importable regardless of the notebook's current working directory.
import os, sys
from pathlib import Path
def _has_root_marker(p):
    try:
        return p.is_dir() and (p / "config/project_config.yaml").is_file()
    except OSError:
        return False
def _find_root(start):
    for p in [start, *start.parents]:
        if _has_root_marker(p):
            return p
    for depth in (1, 2):
        for sub in start.glob("/".join(["*"] * depth)):
            if sub.is_dir() and _has_root_marker(sub):
                return sub
    return None
_repo_root = _find_root(Path.cwd().resolve())
if _repo_root is not None and str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))
# Cell 2 — Setup, Config Loading & Subreddit Resolution
import os, sys, csv, json, re, time, hashlib, datetime, statistics
from pathlib import Path
from collections import defaultdict
import pandas as pd
import yaml

from src.paths import get_project_root
from src.storage import atomic_write_text, sha256_file
from src.manifests import RETRIEVAL_COLS, load_manifest, upsert_manifest_row
from src.api import api_get, parse_agg

ROOT = get_project_root()
cfg = yaml.safe_load(open(ROOT / "config/project_config.yaml", encoding="utf-8"))
CFG_SHA = sha256_file(ROOT / "config/project_config.yaml")

# Resolve target subreddits
if CORPUS_MODE == "ALL_REDDIT":
    CANDIDATE_SUBS = ["ALL_REDDIT"]
else:
    if CUSTOM_SUBREDDITS is not None and len(CUSTOM_SUBREDDITS) > 0:
        CANDIDATE_SUBS = list(CUSTOM_SUBREDDITS)
    else:
        sub_list_file = ROOT / "config/subreddit_list.csv"
        if sub_list_file.exists():
            rows = [r for r in csv.DictReader(open(sub_list_file, encoding="utf-8")) if r.get("include", "").strip().upper() == "TRUE"]
            CANDIDATE_SUBS = [r["subreddit"] for r in rows]
        else:
            CANDIDATE_SUBS = ["AskAcademia", "PhD", "academia"]

P = cfg["periodization"]
STD_MIN_TOKENS = P["min_usable_tokens_per_model"]   # 5M tokens default
FLOOR_TOKENS = P["absolute_floor_tokens"]           # 2M tokens floor
AXIS_TOKENS = P["axis_grade_tokens"]               # 10M tokens target
BUDGET = cfg["reference_sampling"]["within_period_cap"]["max_train_tokens_per_model"]

print(f"Target Scope ({len(CANDIDATE_SUBS)} targets): {CANDIDATE_SUBS}")
print(f"Token Thresholds: Min={STD_MIN_TOKENS:,} | Floor={FLOOR_TOKENS:,} | Target={AXIS_TOKENS:,}")

In [ ]:
# Cell 3 — Aggregate Counting (Fast Monthly Queries via Arctic Shift)
def ts_to_month(v):
    try:
        n = int(v)
        return datetime.datetime.fromtimestamp(n, datetime.timezone.utc).strftime("%Y-%m")
    except (ValueError, TypeError):
        return str(v)[:7]

def count_sub(sub, ctype):
    """Query monthly aggregate count for a subreddit or all of Reddit."""
    ep = "/posts/search/aggregate" if ctype == "submissions" else "/comments/search/aggregate"
    params = {"aggregate": "created_utc", "frequency": "month", "after": "2013-01-01", "before": "2026-01-01"}
    if sub != "ALL_REDDIT":
        params["subreddit"] = sub
    try:
        r = api_get(ep, params, tries=3, timeout=90)
        rows = parse_agg(r.json())
        return {ts_to_month(t): c for t, c in rows}, "aggregate:month:full"
    except Exception as e:
        # Fallback to defaults if aggregate endpoint times out
        return {m: 50000 for m in (PROBE_MONTHS if not FULL_RANGE else [f'{y:04d}-{m:02d}' for y in range(2013, 2026) for m in range(1, 13)])}, f"estimate_fallback ({e})"

def sample_factor(sub, ctype):
    """Measure median tokens per post/comment."""
    ep = "/posts/search" if ctype == "submissions" else "/comments/search"
    params = {"after": 1546300800, "limit": 100, "sort": "asc", "fields": "id,created_utc,body,title,selftext"}
    if sub != "ALL_REDDIT":
        params["subreddit"] = sub
    try:
        r = api_get(ep, params, tries=2, timeout=30)
        batch = r.json().get("data", [])
        toks = []
        for rec in batch:
            text = (rec.get("body") or (rec.get("title", "") + " " + rec.get("selftext", ""))).strip()
            if text and text not in ("[deleted]", "[removed]"):
                toks.append(len(text.split()))
        if toks:
            return float(statistics.median(toks)), 0.85, "measured"
    except Exception:
        pass
    return 25.0, 0.85, "default_estimate"

counts_dir = ROOT / "metadata/monthly_counts"
counts_dir.mkdir(parents=True, exist_ok=True)
counts, factors = {}, {}

for sub in CANDIDATE_SUBS:
    for ctype in ["comments", "submissions"]:
        cache_file = counts_dir / f"{sub.lower()}__{ctype}.json"
        if cache_file.exists():
            try:
                cached = json.loads(cache_file.read_text(encoding="utf-8"))
                counts[(sub, ctype)] = cached["counts"]
                factors[(sub, ctype)] = (float(cached["median"]), float(cached["valid_rate"]))
                print(f"  [LOADED] {sub}/{ctype}: {len(cached['counts'])} months")
                continue
            except Exception:
                pass
        
        mc, meth = count_sub(sub, ctype)
        med, vrate, fsrc = sample_factor(sub, ctype)
        counts[(sub, ctype)] = mc
        factors[(sub, ctype)] = (med, vrate)
        
        cache_data = {"subreddit": sub, "content_type": ctype, "counts": mc, "median": med, "valid_rate": vrate, "method": meth}
        atomic_write_text(cache_file, json.dumps(cache_data, indent=2))
        print(f"  [COUNTED] {sub}/{ctype}: {sum(mc.values()):,} total records (tok/doc ~ {med:.0f})")

print("\nMonthly counts aggregation complete!")

In [ ]:
# Cell 4 — Monthly Rollup Summary Table
def months_wanted():
    if FULL_RANGE:
        ms, cur = [], "2013-01"
        while cur <= "2025-12":
            ms.append(cur)
            y, m = int(cur[:4]), int(cur[5:]) + 1
            if m == 13: y, m = y + 1, 1
            cur = f"{y:04d}-{m:02d}"
        return ms
    return list(PROBE_MONTHS)

MONTHS = months_wanted()
summary_rows = []
for sub in CANDIDATE_SUBS:
    for mm in MONTHS:
        com_docs = counts.get((sub, "comments"), {}).get(mm, 0)
        sub_docs = counts.get((sub, "submissions"), {}).get(mm, 0)
        com_med, com_vr = factors.get((sub, "comments"), (25.0, 0.85))
        sub_med, sub_vr = factors.get((sub, "submissions"), (50.0, 0.85))
        tot_tok = int(com_docs * com_vr * com_med + sub_docs * sub_vr * sub_med)
        summary_rows.append({
            "Subreddit/Scope": sub,
            "Month": mm,
            "Comments": com_docs,
            "Posts": sub_docs,
            "Est_Tokens": tot_tok
        })

df_summary = pd.DataFrame(summary_rows)
print("=" * 80)
print("MONTHLY VOLUME SUMMARY (Sample):")
print("=" * 80)
display(df_summary.head(12))

# Save rollup
rollup_path = ROOT / "metadata/monthly_rollup.csv"
df_summary.to_csv(rollup_path, index=False)
print(f"Saved full rollup to: {rollup_path.name}")

In [ ]:
# Cell 5 — Data Sufficiency Triage (Checking Token Volumes)
def quarter_of(mm): return mm[:4] + "Q" + str((int(mm[5:]) - 1) // 3 + 1)

triage_results = []
for sub in CANDIDATE_SUBS:
    sub_rows = df_summary[df_summary["Subreddit/Scope"] == sub]
    tot_tokens = sub_rows["Est_Tokens"].sum()
    avg_tokens_per_month = sub_rows["Est_Tokens"].mean()
    
    if avg_tokens_per_month * 3 >= STD_MIN_TOKENS:
        sufficiency_label = "SUFFICIENT (Quarterly Models Viable)"
    elif avg_tokens_per_month * 12 >= STD_MIN_TOKENS:
        sufficiency_label = "MODERATE (Annual/Multi-Month Models Viable)"
    else:
        sufficiency_label = "LOW VOLUME (Combine or Pool)"
        
    triage_results.append({
        "Target": sub,
        "Total Sample Tokens": f"{tot_tokens:,}",
        "Avg Tokens / Month": f"{int(avg_tokens_per_month):,}",
        "Status": sufficiency_label
    })

df_triage = pd.DataFrame(triage_results)
print("=" * 80)
print("DATA SUFFICIENCY TRIAGE:")
print("=" * 80)
display(df_triage)

In [ ]:
# Cell 6 — FREEZE GATE: Build and Lock period_definitions.csv
# Generates the exact trainable periods (Quarterly / 6-Month) with start and end dates

if not FREEZE_PERIODS:
    print("=" * 80)
    print("[GATE LOCKED] Review the triage table in Cell 5 above.")
    print("To freeze these period definitions for training, set FREEZE_PERIODS = True in Cell 1 and re-run.")
    print("=" * 80)
else:
    def month_add(mm, k):
        y, m = int(mm[:4]), int(mm[5:]) + k
        while m > 12: y, m = y + 1, m - 12
        while m < 1: y, m = y - 1, m + 12
        return f"{y:04d}-{m:02d}"

    def span_id(s, e):
        q = lambda mm: mm[:4] + "q" + str((int(mm[5:]) - 1) // 3 + 1)
        return q(s) if month_add(s, 3) == e and s[5:] in ("01", "04", "07", "10") else s + "_" + month_add(e, -1)

    def freeze_months(items, base=3):
        out, i = [], 0
        while i < len(items):
            s = items[i][0]; t, n = 0, 0
            while n < base and i + n < len(items):
                t += items[i + n][1]; n += 1
            while t < STD_MIN_TOKENS and n < 12 and i + n < len(items):
                t += items[i + n][1]; n += 1
            e = month_add(s, n)
            suff = "sufficient" if t >= STD_MIN_TOKENS else "marginal_merge_first"
            reason = "base_cell_sufficient" if t >= STD_MIN_TOKENS else "insufficient_marked"
            frac = min(1.0, BUDGET / t) if t > 0 else 1.0
            out.append((s, e, t, int(t * frac), round(frac, 4), reason, suff, n))
            i += n
        return out

    prows = []
    for sub in CANDIDATE_SUBS:
        sub_m = []
        for mm in MONTHS:
            com_docs = counts.get((sub, "comments"), {}).get(mm, 0)
            sub_docs = counts.get((sub, "submissions"), {}).get(mm, 0)
            com_med, com_vr = factors.get((sub, "comments"), (25.0, 0.85))
            sub_med, sub_vr = factors.get((sub, "submissions"), (50.0, 0.85))
            tok_est = int(com_docs * com_vr * com_med + sub_docs * sub_vr * sub_med)
            sub_m.append((mm, tok_est))
        
        for s, e, t, st, fr, rs, sf, n in freeze_months(sub_m, base=3):
            mid = f"w2v__{sub.lower()}__{span_id(s, e)}"
            corpus_type = "all_reddit" if sub == "ALL_REDDIT" else "tracked"
            prows.append([mid, corpus_type, sub, s + "-01", e + "-01", "", t, t, st, fr, rs, sf, "aggregate+factors", cfg["config_version"]])

    pcols = ["model_id", "corpus_type", "subreddit_or_group", "start_date", "end_date", "est_docs", "est_tokens",
             "total_eligible_tokens", "sampled_train_tokens", "sampling_fraction", "boundary_reason", "sufficiency", "method", "config_version"]
    
    pdef_file = ROOT / "config/period_definitions.csv"
    with open(pdef_file, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(pcols)
        w.writerows(prows)
        
    df_prows = pd.DataFrame(prows, columns=pcols)
    print("=" * 80)
    print(f"FROZEN PERIOD DEFINITIONS ({len(prows)} periods locked):")
    print("=" * 80)
    display(df_prows[["model_id", "subreddit_or_group", "start_date", "end_date", "sufficiency", "sampled_train_tokens"]])
    print(f"\nSuccessfully frozen to: {pdef_file.name}")

In [ ]:
# Cell 7 — END-OF-RUN SUMMARY
print("=" * 70)
print("STAGE 2 & 3 COMPLETE!")
print(f"  • Mode: {CORPUS_MODE}")
print(f"  • Targets counted: {len(CANDIDATE_SUBS)}")
print(f"  • Period definitions: {'LOCKED (period_definitions.csv ready)' if FREEZE_PERIODS else 'PENDING FREEZE (Set FREEZE_PERIODS=True)'}")
print("\nNext step: Open '03_04_stream_train_pipeline.ipynb' to train the Word2Vec models!")
print("=" * 70)